# Extracting Trials With Given Properties & Plotting Results

In [2]:
import pandas as pd
import numpy as np
import h5py
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv("Trial_Info.csv")
df

,AnimalCode,Date,TrialID,Time,BFMTime,Duration,Contrast,PerceptualCat,ReactTime,Rewarded?,Enticed?,Seen?,Consume?,Attrition?,Recording,EngagementScore,EngagementScore_S20,MovieFile,Delay
0,cfm001mjr,240506,240506_1_001,3.198,1.841983,0.529,0.060,3,11.880,0,0,0,0,1,1,0.000000,0.000000,N:/GEVI_Wave/Analysis/Visual/cfm001mjr/2024050...,-2.832001
1,cfm001mjr,240506,240506_1_002,6.353,4.996983,0.522,0.023,1,8.725,0,0,0,0,1,1,0.000000,0.000000,N:/GEVI_Wave/Analysis/Visual/cfm001mjr/2024050...,-2.832001
2,cfm001mjr,240506,240506_1_003,10.228,8.871983,0.532,0.000,1,4.850,0,0,0,0,1,1,0.000000,0.057143,N:/GEVI_Wave/Analysis/Visual/cfm001mjr/2024050...,-2.832001
3,cfm001mjr,240506,240506_1_004,14.599,13.242983,0.521,0.024,1,0.479,1,1,1,1,1,1,0.285714,0.086735,N:/GEVI_Wave/Analysis/Visual/cfm001mjr/2024050...,-2.832001
4,cfm001mjr,240506,240506_1_005,43.217,41.860983,0.550,0.023,1,4.464,0,0,0,0,1,1,0.000000,0.130952,N:/GEVI_Wave/Analysis/Visual/cfm001mjr/2024050...,-2.832001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2498,cfm001mjr,240522,240522_1_439,933.166,934.220085,0.539,0.013,1,NaN,0,0,0,0,1,1,0.000000,0.000000,N:/GEVI_Wave/Analysis/Visual/cfm001mjr/2024052...,-0.421899
2499,cfm001mjr,240522,240522_1_441,937.933,938.987085,0.540,0.020,1,NaN,0,0,0,0,1,1,0.000000,0.000000,N:/GEVI_Wave/Analysis/Visual/cfm001mjr/2024052...,-0.421899
2500,cfm001mjr,240522,240522_1_442,939.876,940.930085,0.507,0.045,2,NaN,0,0,0,0,1,1,0.000000,0.000000,N:/GEVI_Wave/Analysis/Visual/cfm001mjr/2024052...,-0.421899
2501,cfm001mjr,240522,240522_1_445,945.859,946.913085,0.509,0.013,1,NaN,0,0,0,0,1,1,0.000000,0.000000,N:/GEVI_Wave/Analysis/Visual/cfm001mjr/2024052...,-0.421899


## Trial Extraction

In [4]:
def find_trials(df, **filters):
    """
    Finds trials from df that match the specified properties.

    Args:
    - df (pd.DataFrame): DataFrame containing trial information.
    - filters (dict): Key-value pairs of column names and values to filter by.
      - Supported keys include 'Date', 'Recording', 'PerceptualCat', 'Rewarded', etc.
      - If a parameter is not specified, it is ignored in filtering.

    Returns:
    - pd.DataFrame: Filtered dataframe containing TrialID, BFMTime, Duration, and MovieFile.
    """

    # Start with full DataFrame and apply filters dynamically
    filtered_df = df.copy()

    for key, value in filters.items():
        if key in df.columns and value is not None:
            filtered_df = filtered_df[filtered_df[key] == value]

    return filtered_df[["TrialID", "BFMTime", "Duration", "MovieFile"]]

def extract_trials_from_movies(trials_df):
    """
    Extracts trials from their corresponding movie files and aligns them in a 4D NumPy array.

    Args:
    - trials_df (pd.DataFrame): Output from `find_trials()`, containing TrialID, BFMTime, Duration, and MovieFile.

    Returns:
    - np.ndarray: 4D array with shape (num_trials, x, y, frames), where each extracted trial is aligned.
    """
    extracted_trials = []  # List to store extracted trials

    # Process each movie file separately
    unique_movie_files = trials_df["MovieFile"].unique()

    for movie_path in unique_movie_files:
        print(f"Processing movie: {movie_path}")

        # Load the movie and extract FPS
        with h5py.File("/Users/katiebrown/Library/CloudStorage/OneDrive-Stanford/Neuro_Project/Data/cG_unmixed_dFF.h5", 'r') as movie_file:
            movie_data = movie_file["mov"][()] 
            fps = movie_file["specs"]["fps"][()][0][0]  

            # Filter trials that belong to this movie
            movie_trials = trials_df[trials_df["MovieFile"] == movie_path]

            # Compute max duration (in frames) for this movie
            durations_frames = (movie_trials["Duration"] * fps).astype(int)
            duration_max_frames = durations_frames.max()
            total_frames_per_trial = int(duration_max_frames * 1.4)  # 40% buffer included
            align_frame = int(0.8 * total_frames_per_trial)  # Stimulus ends at 80% of the trial length

            for _, trial in movie_trials.iterrows():
                bfm_time = trial["BFMTime"]
                duration_frames = int(trial["Duration"] * fps)
                end_frame = int(bfm_time * fps) + duration_frames  # Last frame of stimulus

                # Compute start frame based on alignment
                start_frame = end_frame - align_frame
                start_frame = max(0, start_frame)  # Prevent negative start frames

                # Extract movie segment for this trial
                extracted_movie = movie_data[:, :, start_frame : start_frame + total_frames_per_trial]

                # Ensure extracted movie has correct shape
                if extracted_movie.shape[2] < total_frames_per_trial:
                    # Pad with zeros if the extracted movie is too short
                    pad_width = total_frames_per_trial - extracted_movie.shape[2]
                    extracted_movie = np.pad(
                        extracted_movie,
                        ((0, 0), (0, 0), (0, pad_width)),
                        mode='constant',
                        constant_values=0
                    )

                extracted_trials.append(extracted_movie)

    # Convert list to 4D NumPy array (trial, x, y, frames)
    return np.stack(extracted_trials, axis=0)

In [ ]:
trialsdf = find_trials(df, Date=240506, PerceptualCat=1, Recording=1)
extract_trials_from_movies(trialsdf)